# Method 2 — Transformer-Based Code Classifiers (CodeBERT, GraphCodeBERT, CodeT5)

**Fixes Reviewer #3, M5:** CodeBERT (and the other models below) are
**encoder-based transformer classifiers**, not generative LLMs. A
fine-tuned encoder emits a probability over two fixed classes; it does not
"hallucinate" in the sense that applies to generative models like GPT-style
LLMs. This notebook drops the "LLM" framing and names these accurately.

**Also fixes two real bugs found in the previous version:**
- `DEVICE` was used (`.to(DEVICE)`) but never defined anywhere -- this
  would crash with `NameError` the moment fine-tuning started.
- `VulBERTa` / `LineVul` (the two external baselines Section II-D/IV-F
  promise) turned out to be **C/C++-only** models: VulBERTa uses a custom
  `libclang`-based tokenizer pretrained on C/C++ repositories, and LineVul
  has no published pretrained checkpoint at all -- it's a training recipe
  for fine-tuning CodeBERT on the C/C++ Big-Vul dataset. Applying either
  directly to Python code would be methodologically invalid, not just a
  loading exercise. They are replaced below with two more transformer
  models that are legitimately Python-capable:

| Model | Architecture | Why it's a valid Python baseline |
|---|---|---|
| **CodeBERT** (`microsoft/codebert-base`) | RoBERTa encoder | Pretrained on 6 languages including Python (CodeSearchNet) |
| **GraphCodeBERT** (`microsoft/graphcodebert-base`) | RoBERTa encoder + data-flow pretraining | Same pretraining corpus as CodeBERT, adds structural signal |
| **CodeT5** (`Salesforce/codet5-base`) | T5 encoder-decoder (used here via `T5ForSequenceClassification`) | Pretrained on CodeSearchNet (incl. Python) with an identifier-aware objective |

**Compute note**: fine-tuning three transformer models sequentially is
substantially heavier than one. On CPU-only hardware this can take a long
time -- reduce `EPOCHS` or `MAX_LEN` below if needed, or comment out a
model from the `MODELS` list to run fewer at once.

**Outputs saved (per model):**
- `models/<model_key>_finetuned/` -- fine-tuned weights + tokenizer
- `models/<model_key>_scores.csv` -- per-sample probabilities for every split (id, split, prob)
- `results/llm_metrics.json` -- accuracy/precision/recall/F1/AUC for all three models, keyed by name
- `results/llm_roc_comparison.png` -- combined ROC/AUC curve across all three
- `results/llm_confusion_matrices.png` -- confusion matrices side by side


In [1]:
!pip install torch transformers accelerate scikit-learn pandas numpy matplotlib seaborn tqdm --quiet



[notice] A new release of pip available: 22.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ── Imports ───────────────────────────────────────────────────────────────
import os, json, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from datetime import datetime
from typing import List

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, classification_report, roc_auc_score,
                              roc_curve)

warnings.filterwarnings('ignore')
random.seed(42); np.random.seed(42); torch.manual_seed(42)

os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

# THE FIX: DEVICE was used but never defined in the previous version.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M")}')


c:\Users\Lenda\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
Timestamp: 2026-09-10 18:51


In [3]:
# ── Load dataset ──────────────────────────────────────────────────────────
CSV = 'dataset_with_content.csv'
if not os.path.exists(CSV):
    raise FileNotFoundError('Run the dataset-collection notebook first to generate dataset_with_content.csv')

df = pd.read_csv(CSV)
df.columns = df.columns.str.strip().str.lower()
df = df.dropna(subset=['content', 'label']).reset_index(drop=True)
df['label'] = df['label'].astype(int)
if 'id' not in df.columns:
    df['id'] = df.index

print(f'Dataset: {len(df):,} samples')
print(f'Vulnerable: {(df["label"]==1).sum():,}   Safe: {(df["label"]==0).sum():,}')


Dataset: 14,010 samples
Vulnerable: 2,357   Safe: 11,653


## Models and hyperparameters

Same fine-tuning recipe (epochs, batch size, learning rate) applied
identically to all three models, so the comparison isolates the effect of
the pretrained architecture rather than differing training budgets.


In [4]:
# ── Models to compare ─────────────────────────────────────────────────────
MODELS = [
    {'key': 'codebert',      'name': 'CodeBERT',      'hf_id': 'microsoft/codebert-base'},
    {'key': 'graphcodebert', 'name': 'GraphCodeBERT',  'hf_id': 'microsoft/graphcodebert-base'},
    {'key': 'codet5',        'name': 'CodeT5',         'hf_id': 'Salesforce/codet5-base'},
]

MAX_LEN    = 256
# BATCH_SIZE lowered from 16 -> 8: on CPU-only machines with ~14K samples,
# batch=16 at max_len=256 can push RAM usage high enough for the OS to kill
# the kernel outright (no Python traceback -- this is what "kernel crashed"
# with no error message usually means). Lower further (e.g. 4) or reduce
# MAX_LEN if it still crashes on your machine.
BATCH_SIZE = 8
EPOCHS     = 3
LR         = 2e-5
WARMUP     = 0.1

print('Models to fine-tune:')
for m in MODELS:
    print(f"  - {m['name']} ({m['hf_id']})")
print(f'\nBATCH_SIZE={BATCH_SIZE}, MAX_LEN={MAX_LEN} -- lower these further if you '
      f'hit an unexplained kernel crash (usually means out-of-memory on CPU-only setups).')


Models to fine-tune:
  - CodeBERT (microsoft/codebert-base)
  - GraphCodeBERT (microsoft/graphcodebert-base)
  - CodeT5 (Salesforce/codet5-base)

BATCH_SIZE=8, MAX_LEN=256 -- lower these further if you hit an unexplained kernel crash (usually means out-of-memory on CPU-only setups).


In [5]:
# ── PyTorch Dataset ───────────────────────────────────────────────────────
class CodeDataset(Dataset):
    def __init__(self, codes: List[str], labels: List[int], tokenizer, max_len: int = MAX_LEN):
        self.codes, self.labels = codes, labels
        self.tokenizer, self.max_len = tokenizer, max_len

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.codes[idx], truncation=True, max_length=self.max_len,
            padding='max_length', return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long),
        }


In [6]:
# ── Train / Val / Test split (same seed as every other notebook) ─────────
idx_all = np.arange(len(df))
y_all = df['label'].values

idx_tv, idx_test = train_test_split(idx_all, test_size=0.20, random_state=42, stratify=y_all)
idx_train, idx_val = train_test_split(idx_tv, test_size=0.25, random_state=42, stratify=y_all[idx_tv])

print(f'Train: {len(idx_train):,}   Val: {len(idx_val):,}   Test: {len(idx_test):,}')


Train: 8,406   Val: 2,802   Test: 2,802


## Fine-tune each model in turn

Same training loop for every model regardless of architecture (RoBERTa
encoder for CodeBERT/GraphCodeBERT, T5 encoder-decoder for CodeT5) --
`AutoModelForSequenceClassification` handles the architecture-specific
classification head internally in both cases.


In [ ]:
# ── Fine-tune + evaluate every model ─────────────────────────────────────
import sys, time as _time

def _log(msg):
    """Prints immediately with a timestamp, bypassing any output buffering
    so progress is visible even in environments/execution modes that don't
    flush stdout live by default (e.g. some remote or scripted runs)."""
    elapsed = _time.time() - _RUN_START
    print(f'[{elapsed/60:6.1f} min] {msg}', flush=True)
    sys.stdout.flush()

_RUN_START = _time.time()
all_results = {}   # name -> dict with metrics, probs, preds, trues

_log(f'Starting fine-tuning run: {len(MODELS)} model(s) queued -> '
     f'{", ".join(m["name"] for m in MODELS)}')

SKIP_ALREADY_DONE = True   # set False to force re-training every model from scratch

for m_idx, m in enumerate(MODELS, 1):
    name, hf_id, key = m['name'], m['hf_id'], m['key']
    print(f'\n{"="*70}\n[{m_idx}/{len(MODELS)}] {name} ({hf_id})\n{"="*70}', flush=True)

    # ── Checkpoint resume: if this model already finished in a previous run
    #    (e.g. before a crash on a LATER model), load its saved results
    #    instead of re-training it from scratch. This is what makes a
    #    mid-run kernel crash non-catastrophic. ──────────────────────────
    scores_path = f'models/{key}_scores.csv'
    model_dir = f'models/{key}_finetuned'
    if SKIP_ALREADY_DONE and os.path.exists(scores_path) and os.path.isdir(model_dir):
        _log(f'{name}: found existing results at {scores_path} -- skipping training '
             f'(checkpoint resume). Set SKIP_ALREADY_DONE=False above to force redo.')
        _scores_existing = pd.read_csv(scores_path)
        _test_scores = _scores_existing[_scores_existing['split'] == 'test']
        _merged = _test_scores.merge(df[['id', 'label']], on='id', how='inner')
        test_probs = _merged[f'{key}_prob'].values
        test_trues = _merged['label'].values
        test_preds = (test_probs >= 0.5).astype(int)
        acc = accuracy_score(test_trues, test_preds)
        prec = precision_score(test_trues, test_preds, zero_division=0)
        rec = recall_score(test_trues, test_preds, zero_division=0)
        f1 = f1_score(test_trues, test_preds, zero_division=0)
        auc_val = roc_auc_score(test_trues, test_probs)
        cm = confusion_matrix(test_trues, test_preds)
        _log(f'{name} loaded from checkpoint: Acc={acc:.4f}  F1={f1:.4f}  AUC={auc_val:.4f}')
        all_results[name] = {
            'key': key, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'auc': auc_val,
            'confusion_matrix': cm, 'test_probs': test_probs, 'test_trues': test_trues,
            'train_losses': [], 'val_accs': [], 'val_f1s': [],
        }
        continue

    _log(f'Downloading/loading tokenizer for {name} ({hf_id}) -- first run for a '
         f'model can take a while if it needs to download from Hugging Face...')
    tokenizer = AutoTokenizer.from_pretrained(hf_id)
    _log(f'Tokenizer ready for {name}.')

    train_ds = CodeDataset(df['content'].iloc[idx_train].tolist(), y_all[idx_train].tolist(), tokenizer)
    val_ds   = CodeDataset(df['content'].iloc[idx_val].tolist(),   y_all[idx_val].tolist(),   tokenizer)
    test_ds  = CodeDataset(df['content'].iloc[idx_test].tolist(),  y_all[idx_test].tolist(),  tokenizer)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    _log(f'Downloading/loading pretrained weights for {name} -- this is the slow step '
         f'on first run (can be several hundred MB to ~1 GB depending on the model).')
    model = AutoModelForSequenceClassification.from_pretrained(hf_id, num_labels=2).to(DEVICE)
    _log(f'{name} model loaded on {DEVICE}. Starting training...')
    # T5-based models need these set for the classification head's EOS-pooling
    # and internal decoder-input shifting; usually already set on the
    # published checkpoint's config, but enforce it defensively here.
    if getattr(model.config, 'decoder_start_token_id', None) is None:
        model.config.decoder_start_token_id = tokenizer.pad_token_id
    if getattr(model.config, 'eos_token_id', None) is None and tokenizer.eos_token_id is not None:
        model.config.eos_token_id = tokenizer.eos_token_id

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * WARMUP), num_training_steps=total_steps)
    loss_fn = nn.CrossEntropyLoss()

    train_losses, val_accs, val_f1s = [], [], []
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0.0
        for batch in tqdm(train_loader, desc=f'{name} epoch {epoch+1}/{EPOCHS} train'):
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            lbls = batch['label'].to(DEVICE)
            optimizer.zero_grad()
            out = model(input_ids=ids, attention_mask=mask)
            loss = loss_fn(out.logits, lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()
        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)
        _log(f'{name} epoch {epoch+1}/{EPOCHS} training pass complete (avg loss={avg_loss:.4f}), '
             f'now running validation...')

        model.eval()
        vp, vt = [], []
        with torch.no_grad():
            for batch in val_loader:
                ids  = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                lbls = batch['label'].numpy()
                out = model(input_ids=ids, attention_mask=mask)
                vp.extend(out.logits.argmax(dim=1).cpu().numpy())
                vt.extend(lbls)
        vacc, vf1 = accuracy_score(vt, vp), f1_score(vt, vp, zero_division=0)
        val_accs.append(vacc); val_f1s.append(vf1)
        _log(f'{name} epoch {epoch+1}/{EPOCHS} DONE: loss={avg_loss:.4f}  '
             f'val_acc={vacc:.4f}  val_f1={vf1:.4f}')

    _log(f'{name} training complete. Saving model + scoring test set...')
    os.makedirs(f'models/{key}_finetuned', exist_ok=True)
    model.save_pretrained(f'models/{key}_finetuned')
    tokenizer.save_pretrained(f'models/{key}_finetuned')

    # ── Score every split (for Hybrid.ipynb / app.py, same pattern as Bandit/RL) ──
    def score_loader(loader):
        model.eval()
        probs_out, preds_out, trues_out = [], [], []
        with torch.no_grad():
            for batch in loader:
                ids  = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                lbls = batch['label'].numpy()
                out = model(input_ids=ids, attention_mask=mask)
                prbs = torch.softmax(out.logits, dim=1).cpu().numpy()
                probs_out.extend(prbs[:, 1])
                preds_out.extend(out.logits.argmax(dim=1).cpu().numpy())
                trues_out.extend(lbls)
        return np.array(probs_out), np.array(preds_out), np.array(trues_out)

    train_probs, _, _ = score_loader(train_loader)
    val_probs, _, _ = score_loader(val_loader)
    test_probs, test_preds, test_trues = score_loader(test_loader)

    scores_df = pd.concat([
        pd.DataFrame({'id': df['id'].iloc[idx_train].values, 'split': 'train', f'{key}_prob': train_probs}),
        pd.DataFrame({'id': df['id'].iloc[idx_val].values,   'split': 'val',   f'{key}_prob': val_probs}),
        pd.DataFrame({'id': df['id'].iloc[idx_test].values,  'split': 'test',  f'{key}_prob': test_probs}),
    ], ignore_index=True)
    scores_df.to_csv(f'models/{key}_scores.csv', index=False)

    acc = accuracy_score(test_trues, test_preds)
    prec = precision_score(test_trues, test_preds, zero_division=0)
    rec = recall_score(test_trues, test_preds, zero_division=0)
    f1 = f1_score(test_trues, test_preds, zero_division=0)
    auc_val = roc_auc_score(test_trues, test_probs)
    cm = confusion_matrix(test_trues, test_preds)

    print(f'\n=== {name} — Test Set ===')
    print(f'Accuracy={acc:.4f}  Precision={prec:.4f}  Recall={rec:.4f}  F1={f1:.4f}  AUC={auc_val:.4f}')

    all_results[name] = {
        'key': key, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'auc': auc_val,
        'confusion_matrix': cm, 'test_probs': test_probs, 'test_trues': test_trues,
        'train_losses': train_losses, 'val_accs': val_accs, 'val_f1s': val_f1s,
    }

    _log(f'[{m_idx}/{len(MODELS)}] {name} fully done -- '
         f'Acc={acc:.4f} F1={f1:.4f} AUC={auc_val:.4f}')
    # Explicit memory cleanup between models -- important on CPU-only, low-RAM
    # setups where three transformer models loaded in sequence without this
    # can accumulate enough memory pressure to crash the kernel.
    del model, tokenizer, train_loader, val_loader, test_loader, train_ds, val_ds, test_ds
    import gc
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    _log(f'Memory cleaned up after {name}.')

_log('All models fine-tuned and evaluated.')
print('\nAll models fine-tuned and evaluated.')


[   0.0 min] Starting fine-tuning run: 3 model(s) queued -> CodeBERT, GraphCodeBERT, CodeT5

[1/3] CodeBERT (microsoft/codebert-base)
[   0.0 min] Downloading/loading tokenizer for CodeBERT (microsoft/codebert-base) -- first run for a model can take a while if it needs to download from Hugging Face...
[   0.0 min] Tokenizer ready for CodeBERT.
[   0.0 min] Downloading/loading pretrained weights for CodeBERT -- this is the slow step on first run (can be several hundred MB to ~1 GB depending on the model).


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[   0.1 min] CodeBERT model loaded on cpu. Starting training...


CodeBERT epoch 1/3 train: 100%|██████████| 1051/1051 [5:54:55<00:00, 20.26s/it] 

[ 355.0 min] CodeBERT epoch 1/3 training pass complete (avg loss=0.2807), now running validation...


...

[1133.0 min] GraphCodeBERT model loaded on cpu. Starting training...


GraphCodeBERT epoch 1/3 train: 100%|██████████| 1051/1051 [6:55:16<00:00, 23.71s/it] 

[1548.2 min] GraphCodeBERT epoch 1/3 training pass complete (avg loss=0.2616), now running validation...


[1582.0 min] GraphCodeBERT epoch 1/3 DONE: loss=0.2616  val_acc=0.9154  val_f1=0.7481


GraphCodeBERT epoch 2/3 train:  32%|███▏      | 336/1051 [2:02:39<7:18:56, 36.83s/it]

## Metrics comparison table


In [ ]:
# ── Metrics comparison table ──────────────────────────────────────────────
comparison_df = pd.DataFrame([
    {'Model': name, 'Accuracy': r['accuracy'], 'Precision': r['precision'],
     'Recall': r['recall'], 'F1': r['f1'], 'AUC': r['auc']}
    for name, r in all_results.items()
]).set_index('Model')

print(comparison_df.round(4).to_string())


## Combined ROC/AUC comparison

Direct visual comparison of all three transformer models on the same test
set -- a curve further toward the top-left with higher AUC discriminates
vulnerable from safe code better across every threshold.


In [ ]:
# ── Combined ROC curve + confusion matrices ──────────────────────────────
BG, AXB, CT, CG = '#0f172a', '#1e293b', '#f1f5f9', '#334155'
MODEL_COLORS = {'CodeBERT': '#3b82f6', 'GraphCodeBERT': '#22c55e', 'CodeT5': '#a855f7'}

fig, ax = plt.subplots(figsize=(7, 7), facecolor=BG)
ax.set_facecolor(AXB)
for name, r in all_results.items():
    fpr, tpr, _ = roc_curve(r['test_trues'], r['test_probs'])
    ax.plot(fpr, tpr, linewidth=2.5, color=MODEL_COLORS.get(name),
            label=f"{name} (AUC = {r['auc']:.4f})")
ax.plot([0, 1], [0, 1], linestyle='--', color=CG, label='Random guess (AUC = 0.50)')
ax.set_xlabel('False Positive Rate', color=CT)
ax.set_ylabel('True Positive Rate', color=CT)
ax.set_title('Transformer Model Comparison — ROC Curves', color=CT, fontweight='bold')
ax.tick_params(colors=CT)
ax.legend(loc='lower right', facecolor=AXB, labelcolor=CT)
ax.grid(alpha=0.2, color=CG)
plt.tight_layout()
plt.savefig('results/llm_roc_comparison.png', dpi=300, facecolor=BG, bbox_inches='tight')
plt.show()
print('Saved: results/llm_roc_comparison.png')

fig, axes = plt.subplots(1, len(all_results), figsize=(6*len(all_results), 5), facecolor=BG)
if len(all_results) == 1:
    axes = [axes]
for ax, (name, r) in zip(axes, all_results.items()):
    ax.set_facecolor(AXB)
    sns.heatmap(r['confusion_matrix'], annot=True, fmt='d', ax=ax, cmap='RdYlGn',
                xticklabels=['Safe', 'Vulnerable'], yticklabels=['Safe', 'Vulnerable'],
                cbar=False, annot_kws={'size': 14, 'color': 'white'})
    ax.set_title(name, color=CT, fontweight='bold')
    ax.set_xlabel('Predicted', color='#94a3b8')
    ax.set_ylabel('Actual', color='#94a3b8')
    ax.tick_params(colors='#94a3b8')
plt.tight_layout()
plt.savefig('results/llm_confusion_matrices.png', dpi=300, facecolor=BG, bbox_inches='tight')
plt.show()
print('Saved: results/llm_confusion_matrices.png')


In [ ]:
# ── Save metrics ────────────────────────────────────────────────────────
metrics_out = {}
for name, r in all_results.items():
    tn, fp, fn, tp = r['confusion_matrix'].ravel()
    metrics_out[name] = {
        'hf_id': next(m['hf_id'] for m in MODELS if m['name'] == name),
        'accuracy': round(float(r['accuracy']), 4),
        'precision': round(float(r['precision']), 4),
        'recall': round(float(r['recall']), 4),
        'f1': round(float(r['f1']), 4),
        'auc': round(float(r['auc']), 4),
        'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)},
        'training': {'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'max_len': MAX_LEN, 'lr': LR,
                     'train_losses': r['train_losses'], 'val_accs': r['val_accs'], 'val_f1s': r['val_f1s']},
    }

with open('results/llm_metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)

print('Saved: results/llm_metrics.json')
print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk != 'training'} for k, v in metrics_out.items()}, indent=2))


## Next steps

1. Update Section IV-B / II-B of the paper: replace "LLM Module" with
   "Transformer-based code classifier module," and drop hallucination
   framing for these encoder/encoder-decoder models specifically.
2. Update Section II-D / IV-F: VulBERTa and LineVul are C/C++-specific and
   were replaced with GraphCodeBERT and CodeT5 for a methodologically
   valid same-dataset, same-language comparison; state this explicitly
   rather than silently substituting.
3. `Hybrid.ipynb` should pick the best-performing model here (by AUC) as
   the "semantic module" input, loading `models/<key>_scores.csv`.
4. Extend `app.py`'s `METHOD_REGISTRY` with entries for GraphCodeBERT and
   CodeT5 (currently only has one generic `CodeBERT` slot) so the
   dashboard shows all three automatically.
